# Lesson 2: What is Generative AI? 🤖
**Learning goals**
1. Tell the difference between **discriminative AI** (it judges) and **generative AI** (it creates)
2. Learn the most basic Python: variables, lists, loops
3. Run a "judging" AI and a "creating" AI yourself

**⏱️ Timing (60 min):** Python warm-up 15' | random sentence generator 10' | discriminative AI 10' | generative AI 15' | quiz 10'

> ⚙️ **Before you start:** go to **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

## 🍎 An analogy first
| | Discriminative AI | Generative AI |
|---|---|---|
| It's like… | **A teacher marking tests**: looks at an answer and says right or wrong | **An author or artist**: writes a new essay, paints a new picture |
| Input → output | a photo → "cat" | "draw a cat wearing a hat" → a brand-new picture |
| Everyday examples | Face ID, spam filters, photo apps sorting your pictures | ChatGPT / Claude, AI art, AI music |

One sentence to remember: **discriminative AI answers "what is this?", generative AI answers "make me one".**

## Part 1: Python warm-up 🐍
In Colab, click the ▶️ button on the left of a cell to run it. Try it!

In [ ]:
# print makes the computer "talk"
print("Hello, world of AI!")

# A variable is like a labelled box that holds something
name = "Alex"      # 👉 change this to your name
age = 13
print(f"My name is {name} and I am {age} years old")

In [ ]:
# A list is a row of boxes; a for loop opens each box one by one
hobbies = ["basketball", "drawing", "gaming"]
for h in hobbies:
    print("I like", h)

### 🧩 Exercise 1
Replace `hobbies` with 5 of your own hobbies, then add one line that prints **how many hobbies** you have (hint: `len(hobbies)`).

In [ ]:
# Write your code here

## Part 2: The simplest possible "generator" 🎲
At its heart, AI text generation means **picking one option out of many**. Let's build the dumbest version with dice (random numbers).

In [ ]:
import random

who  = ["A cat", "An alien", "My maths teacher", "A robot"]
did  = ["stole", "invented", "sang a song about", "fell in love with"]
what = ["the moon", "a bowl of noodles", "my homework", "a volcano"]

for i in range(5):
    print(random.choice(who), random.choice(did), random.choice(what))

### 🧩 Exercise 2
1. Add 3 of your own words to each list and run it a few times.
2. **Think:** does this generator "understand" what it's saying? What is the biggest difference between it and ChatGPT?
(Hint: it is completely random. It never **learned** from data which words go well together — that's what the next few lessons are about.)

## Part 3: Discriminative AI — the mood detector 😊😡
The model below read tens of thousands of movie reviews and learned to judge whether a sentence is **positive** or **negative**. It can only judge; it can't write new sentences.

In [ ]:
from transformers import pipeline
judge = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

sentences = [
    "I love this movie, it is amazing!",
    "This pizza is terrible.",
    "The homework was not as bad as I thought.",
]
for s in sentences:
    r = judge(s)[0]
    print(f"{r['label']:8} ({r['score']:.0%})  ←  {s}")

### 🧩 Exercise 3: Trick the AI
Write 3 **tricky** sentences and see if the AI gets them wrong: sarcasm ("Oh great, another Monday."), double negatives, mixed feelings. Record the ones it gets wrong.

In [ ]:
my_sentences = [
    "Oh great, another Monday.",
    # add your sentences here
]
for s in my_sentences:
    print(judge(s)[0], s)

## Part 4: Generative AI — a model that writes ✍️
Now let's load a real generative model (Qwen, a small open language model).

In [ ]:
# ⏳ The first run downloads the model (about 1–3 minutes). Be patient!
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct" if torch.cuda.is_available() else "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
llm = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto", device_map="auto")

def ask(prompt, system="You are a friendly assistant. Answer in simple English.",
        max_new_tokens=300, temperature=0.7):
    """Send a prompt to the AI and return its answer."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens,
                       do_sample=temperature > 0, temperature=temperature if temperature > 0 else None)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("✅ Model loaded:", MODEL)

In [ ]:
print(ask("Write a 3-sentence story about a cat who wants to go to the moon."))

### 🧩 Exercise 4
1. Ask the AI to write a 4-line poem about your school.
2. Run the **same prompt twice**. Are the answers the same? Why not? (We'll find out in Lesson 5.)

### 🚀 Challenge (senior students)
Make the AI **generate** a movie review, then use `judge` to **classify** it. You've just connected generative and discriminative AI!

In [ ]:
review = ask("Write one short, very negative sentence about a movie.", max_new_tokens=40)
print("AI wrote:  ", review)
print("AI judged: ", judge(review)[0])

## Part 5: Quiz 📝
Label each app `"discriminative"` or `"generative"`, then run to see your score.

In [ ]:
my_answers = {
    "Face ID unlocking a phone": "",
    "AI drawing a profile picture": "",
    "Spam email filter": "",
    "ChatGPT writing an essay": "",
    "AI writing a caption for a photo": "",
    "Detecting a broken bone in an X-ray": "",
}
correct = {"Face ID unlocking a phone": "discriminative", "AI drawing a profile picture": "generative",
           "Spam email filter": "discriminative", "ChatGPT writing an essay": "generative",
           "AI writing a caption for a photo": "generative", "Detecting a broken bone in an X-ray": "discriminative"}
score = sum(my_answers[k].strip().lower() == v for k, v in correct.items())
print(f"Score: {score}/{len(correct)}")
for k, v in correct.items():
    if my_answers[k].strip().lower() != v: print("❌", k, "→ correct answer:", v)

## 📌 Summary
- Discriminative AI **judges** (what is this?); generative AI **creates** (make me one)
- Random generation is easy; generating something **meaningful** is hard — that requires **learning** from data
- **Homework:** write down "How do I think AI learns to write?" Keep it — we'll look at it again in Lesson 17.